<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Claude API for Python Developers</h1>
<h1>4. Tools</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
import anthropic
import json
from IPython.display import display, Markdown

import matplotlib.pyplot as plt

import watermark

%load_ext watermark
%matplotlib inline

We start by print out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.13.3
IPython version      : 9.8.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.1.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 87485bac20625f43da5809d38ec119aca4321f71

watermark : 2.5.0
IPython   : 9.8.0
anthropic : 0.75.0
json      : 2.0.9
matplotlib: 3.10.7



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

Initialize the client

In [4]:
client = anthropic.Anthropic()

And define our utility functions

In [5]:
def show_response(response):
    """Display Claude's response with markdown formatting."""
    display(Markdown(response.content[0].text))

def show_json(content):
    print(json.dumps(content, indent=2))

## Tools

Declare the weather tool

In [6]:
weather_tool = {
    "name": "get_weather",
    "description": "Get the current weather for a specific location. Use this when the user asks about weather conditions.",
    "input_schema": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city and country, e.g., 'London, UK' or 'New York, US'"
            },
            "unit": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "Temperature unit preference"
            }
        },
        "required": ["location"]
    }
}

In [7]:
show_json(weather_tool)

{
  "name": "get_weather",
  "description": "Get the current weather for a specific location. Use this when the user asks about weather conditions.",
  "input_schema": {
    "type": "object",
    "properties": {
      "location": {
        "type": "string",
        "description": "The city and country, e.g., 'London, UK' or 'New York, US'"
      },
      "unit": {
        "type": "string",
        "enum": [
          "celsius",
          "fahrenheit"
        ],
        "description": "Temperature unit preference"
      }
    },
    "required": [
      "location"
    ]
  }
}


Make a request with the tool

In [8]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[weather_tool],
    messages=[
        {"role": "user", "content": "What's the weather like in Tokyo?"}
    ]
)

Claude returns multiple content blocks:

In [9]:
response.content

[TextBlock(citations=None, text="I'll check the current weather in Tokyo for you.", type='text'),
 ToolUseBlock(id='toolu_01Jv8vBhBnJ11xRGWj8z36GW', input={'location': 'Tokyo, Japan'}, name='get_weather', type='tool_use')]

And signals that it wants to call a tool in the `stop_reason` field

In [10]:
response.stop_reason

'tool_use'

In [11]:
print("\nContent blocks:")
for block in response.content:
    print(f"  Type: {block.type}")
    if block.type == "tool_use":
        print(f"  Tool: {block.name}")
        print(f"  Input: {json.dumps(block.input, indent=2)}")
        print(f"  Tool Use ID: {block.id}")


Content blocks:
  Type: text
  Type: tool_use
  Tool: get_weather
  Input: {
  "location": "Tokyo, Japan"
}
  Tool Use ID: toolu_01Jv8vBhBnJ11xRGWj8z36GW


The Tool Use ID allows us to identify which tool call the results refer to

Mock tool results.

In [12]:
def get_weather(location, unit="celsius"):
    """Simulated weather function."""
    # In real app, this would call a weather API
    weather_data = {
        "location": location,
        "temperature": 22 if unit == "celsius" else 72,
        "unit": unit,
        "condition": "Partly cloudy",
        "humidity": 65
    }
    return weather_data

Find the tool use request

In [13]:
tool_use_block = next(block for block in response.content if block.type == "tool_use")

Execute the tool

In [14]:
tool_result = get_weather(**tool_use_block.input)


Send tool result back to Claude for final response

In [15]:
final_response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[weather_tool],
    messages=[
        {"role": "user", "content": "What's the weather like in Tokyo?"},
        {"role": "assistant", "content": response.content},
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result", # <===
                    "tool_use_id": tool_use_block.id, # <===
                    "content": json.dumps(tool_result)
                }
            ]
        }
    ]
)

In [16]:
show_response(final_response)

The current weather in Tokyo, Japan is:
- **Temperature**: 22°C (72°F)
- **Condition**: Partly cloudy
- **Humidity**: 65%

It's a pleasant day in Tokyo with mild temperatures and partly cloudy skies!

---
## Structured Outputs

Tools are powerful for forcing Claude to output data in specific formats.

Define a tool for structured entity extraction by asking Claude to provide the inputs we need

In [17]:
entity_extraction_tool = {
    "name": "extract_entities",
    "description": "Extract structured information from text including people, organizations, locations, and dates.",
    "input_schema": {
        "type": "object",
        "properties": {
            "people": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string"},
                        "role": {"type": "string"}
                    },
                    "required": ["name"]
                },
                "description": "People mentioned in the text"
            },
            "organizations": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Organizations mentioned"
            },
            "locations": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Geographic locations mentioned"
            },
            "dates": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Dates or time periods mentioned"
            }
        },
        "required": ["people", "organizations", "locations", "dates"]
    }
}

Sample text

In [18]:
text = """
Apple CEO Tim Cook announced at the WWDC conference in San Francisco on June 5, 2024 
that the company would be partnering with OpenAI. The collaboration was praised by 
Microsoft's Satya Nadella, who attended the event alongside Google's Sundar Pichai.
"""

Try it out!

In [19]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[entity_extraction_tool],
    tool_choice={"type": "tool", "name": "extract_entities"},  # Force tool use
    messages=[
        {"role": "user", "content": f"Extract all entities from this text:\n\n{text}"}
    ]
)

# Parse the structured output
tool_use = next(block for block in response.content if block.type == "tool_use")
entities = tool_use.input

In [20]:
show_json(entities)

{
  "people": [
    {
      "name": "Tim Cook",
      "role": "Apple CEO"
    },
    {
      "name": "Satya Nadella",
      "role": "Microsoft CEO"
    },
    {
      "name": "Sundar Pichai",
      "role": "Google CEO"
    }
  ],
  "organizations": [
    "Apple",
    "OpenAI",
    "Microsoft",
    "Google"
  ],
  "locations": [
    "San Francisco"
  ],
  "dates": [
    "June 5, 2024"
  ]
}


Structured output for sentiment analysis

In [21]:
sentiment_tool = {
    "name": "analyze_sentiment",
    "description": "Analyze the sentiment of a text and return structured results.",
    "input_schema": {
        "type": "object",
        "properties": {
            "overall_sentiment": {
                "type": "string",
                "enum": ["very_positive", "positive", "neutral", "negative", "very_negative"],
                "description": "The overall sentiment of the text"
            },
            "confidence": {
                "type": "number",
                "minimum": 0,
                "maximum": 1,
                "description": "Confidence score between 0 and 1"
            },
            "key_phrases": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "phrase": {"type": "string"},
                        "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]}
                    },
                    "required": ["phrase", "sentiment"]
                },
                "description": "Key phrases that influenced the sentiment"
            },
            "summary": {
                "type": "string",
                "description": "Brief explanation of the sentiment analysis"
            }
        },
        "required": ["overall_sentiment", "confidence", "key_phrases", "summary"]
    }
}

Analyze customer reviews

In [22]:
reviews = [
    "The product quality is amazing! Fast shipping and great customer service. Highly recommend!",
    "It's okay, nothing special. Does what it's supposed to do but overpriced.",
    "Terrible experience. Product arrived damaged and support was unhelpful. Never again!"
]

Get sentiment analysis results form Claude

In [23]:
for review in reviews:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        tools=[sentiment_tool],
        tool_choice={"type": "tool", "name": "analyze_sentiment"},
        messages=[
            {"role": "user", "content": f"Analyze the sentiment of this review:\n\n{review}"}
        ]
    )
    
    tool_use = next(block for block in response.content if block.type == "tool_use")
    result = tool_use.input
    
    # Map sentiment to emoji
    sentiment_emoji = {
        "very_positive": "🌟", "positive": "😊", 
        "neutral": "😐", "negative": "😞", "very_negative": "😡"
    }
    
    print(f"Review: \"{review[:50]}...\"")
    print(f"  {sentiment_emoji.get(result['overall_sentiment'], '❓')} {result['overall_sentiment'].replace('_', ' ').title()}")
    print(f"  Confidence: {result['confidence']:.0%}")
    print(f"  Summary: {result['summary']}")
    print()

Review: "The product quality is amazing! Fast shipping and ..."
  🌟 Very Positive
  Confidence: 95%
  Summary: This review expresses extremely positive sentiment with multiple strong positive indicators including praise for product quality, shipping speed, customer service, and an explicit recommendation. The language is enthusiastic and contains no negative elements.

Review: "It's okay, nothing special. Does what it's suppose..."
  😞 Negative
  Confidence: 75%
  Summary: The review expresses a lukewarm to negative sentiment. While the reviewer acknowledges the product functions as expected, they emphasize that it's unremarkable and costs too much, leading to an overall dissatisfied tone.

Review: "Terrible experience. Product arrived damaged and s..."
  😡 Very Negative
  Confidence: 95%
  Summary: This review expresses extreme dissatisfaction with multiple aspects of the experience - product quality, delivery condition, and customer service. The language is strongly negative througho

## Choosing the Right Tool

When there are multiple tools available, Claude chooses the right one

In [24]:
tools = [
    {
        "name": "calculate",
        "description": "Perform mathematical calculations. Use for arithmetic, percentages, etc.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The mathematical expression to evaluate, e.g., '(10 + 5) * 2'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "convert_currency",
        "description": "Convert between currencies using current exchange rates.",
        "input_schema": {
            "type": "object",
            "properties": {
                "amount": {"type": "number", "description": "Amount to convert"},
                "from_currency": {"type": "string", "description": "Source currency code (USD, EUR, etc.)"},
                "to_currency": {"type": "string", "description": "Target currency code"}
            },
            "required": ["amount", "from_currency", "to_currency"]
        }
    },
    {
        "name": "get_stock_price",
        "description": "Get the current stock price for a given ticker symbol.",
        "input_schema": {
            "type": "object",
            "properties": {
                "symbol": {"type": "string", "description": "Stock ticker symbol (e.g., AAPL, GOOGL)"}
            },
            "required": ["symbol"]
        }
    }
]

Use different queries to see how Claude picks the right tool

In [25]:
test_queries = [
    "What is 15% of 200?",
    "Convert 100 USD to EUR",
    "What's the current price of Apple stock?"
]

for query in test_queries:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        tools=tools,
        messages=[{"role": "user", "content": query}]
    )
    
    print(f"Query: {query}")
    for block in response.content:
        if block.type == "tool_use":
            print(f"  → Claude chose: {block.name}")
            print(f"  → Parameters: {json.dumps(block.input)}")
    print()

Query: What is 15% of 200?
  → Claude chose: calculate
  → Parameters: {"expression": "15/100 * 200"}

Query: Convert 100 USD to EUR
  → Claude chose: convert_currency
  → Parameters: {"amount": 100, "from_currency": "USD", "to_currency": "EUR"}

Query: What's the current price of Apple stock?
  → Claude chose: get_stock_price
  → Parameters: {"symbol": "AAPL"}



### Tool choice options

#### Claude Decides

In [26]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    tools=tools,
    tool_choice={"type": "auto"},  # Default behavior
    messages=[{"role": "user", "content": "Hello, how are you?"}]
)
print(f"   Stop reason: {response.stop_reason}")
print(f"   Response: {response.content[0].text if response.content[0].type == 'text' else 'Tool use'}")

   Stop reason: end_turn
   Response: Hello! I'm doing well, thank you for asking. I'm here and ready to help you with any questions or tasks you might have. 

I have access to some useful tools for mathematical calculations, currency conversions, and stock price lookups if you need assistance with any of those. Is there anything specific I can help you with today?


#### Force tool use


In [27]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    tools=tools,
    tool_choice={"type": "any"},
    messages=[{"role": "user", "content": "I want to know something about money"}]
)
tool_block = next((b for b in response.content if b.type == "tool_use"), None)
if tool_block:
    print(f"   Tool used: {tool_block.name}")

   Tool used: get_stock_price


#### Force specific tool

In [28]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    tools=tools,
    tool_choice={"type": "tool", "name": "calculate"},
    messages=[{"role": "user", "content": "What's the weather like?"}]  # Irrelevant query
)
tool_block = next((b for b in response.content if b.type == "tool_use"), None)
if tool_block:
    print(f"   Forced tool: {tool_block.name}")
    print(f"   Input: {tool_block.input}")

   Forced tool: calculate
   Input: {'expression': '1 + 1'}


## Agents

Define a set of tools for a simple assistant

In [29]:
assistant_tools = [
    {
        "name": "search_database",
        "description": "Search the product database for items matching a query.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"},
                "category": {
                    "type": "string",
                    "enum": ["electronics", "clothing", "books", "all"],
                    "description": "Product category to search"
                },
                "max_results": {"type": "integer", "default": 5}
            },
            "required": ["query"]
        }
    },
    {
        "name": "get_product_details",
        "description": "Get detailed information about a specific product.",
        "input_schema": {
            "type": "object",
            "properties": {
                "product_id": {"type": "string", "description": "The product ID"}
            },
            "required": ["product_id"]
        }
    },
    {
        "name": "add_to_cart",
        "description": "Add a product to the shopping cart.",
        "input_schema": {
            "type": "object",
            "properties": {
                "product_id": {"type": "string"},
                "quantity": {"type": "integer", "default": 1}
            },
            "required": ["product_id"]
        }
    },
    {
        "name": "check_inventory",
        "description": "Check if a product is in stock.",
        "input_schema": {
            "type": "object",
            "properties": {
                "product_id": {"type": "string"}
            },
            "required": ["product_id"]
        }
    }
]

Mock implementations

In [30]:
def execute_tool(name, inputs):
    """Execute a tool and return simulated results."""
    if name == "search_database":
        return {
            "results": [
                {"id": "P001", "name": "Wireless Headphones", "price": 79.99},
                {"id": "P002", "name": "Bluetooth Speaker", "price": 49.99},
                {"id": "P003", "name": "USB-C Hub", "price": 34.99}
            ]
        }
    elif name == "get_product_details":
        return {
            "id": inputs["product_id"],
            "name": "Wireless Headphones",
            "price": 79.99,
            "description": "High-quality wireless headphones with noise cancellation",
            "rating": 4.5,
            "reviews": 1250
        }
    elif name == "add_to_cart":
        return {"success": True, "message": f"Added {inputs.get('quantity', 1)} item(s) to cart"}
    elif name == "check_inventory":
        return {"in_stock": True, "quantity_available": 50}
    return {"error": "Unknown tool"}

### Define basic Agent

In [31]:
def run_agent(user_message, tools, max_iterations=5):
    messages = [{"role": "user", "content": user_message}]
    
    print(f"User: {user_message}")
    print("-" * 50)
    
    for iteration in range(max_iterations):
        # Query Claude
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        
        # Check for tool use
        if response.stop_reason == "tool_use":
            # Process each tool use
            tool_results = []
            
            for block in response.content:
                if block.type == "tool_use":
                    print(f" Tool: {block.name}")
                    print(f" Input: {json.dumps(block.input)}")
                    
                    # Execute the tool
                    result = execute_tool(block.name, block.input)
                    print(f" Result: {json.dumps(result)[:100]}...")
                    
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)
                    })
            
            # Add assistant response and tool results to messages
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})
            
        else:
            # Claude is done with tools, return final response
            final_text = next(
                (block.text for block in response.content if block.type == "text"),
                "No response"
            )
            print("-" * 50)
            print(f" Assistant: {final_text}")
            return final_text
    
    return "Max iterations reached"

Run the agent

In [32]:
result = run_agent(
    "I'm looking for wireless audio devices. Can you show me what's available?",
    assistant_tools
)

User: I'm looking for wireless audio devices. Can you show me what's available?
--------------------------------------------------
 Tool: search_database
 Input: {"query": "wireless audio devices", "category": "electronics"}
 Result: {"results": [{"id": "P001", "name": "Wireless Headphones", "price": 79.99}, {"id": "P002", "name": "...
--------------------------------------------------
 Assistant: Based on the search results, here are the wireless audio devices currently available:

1. **Wireless Headphones** (Product ID: P001) - $79.99
2. **Bluetooth Speaker** (Product ID: P002) - $49.99

I also found a USB-C Hub in the results, but that's not specifically an audio device.

Would you like me to get more detailed information about either of these wireless audio products, such as specifications, features, or availability? I can also check if they're currently in stock or help you add them to your cart if you're interested.


Multi-step task

In [33]:
print("\n" + "=" * 60)
print("MULTI-STEP TASK")
print("=" * 60 + "\n")

result = run_agent(
    "Can you find wireless headphones, check if they're in stock, and add 2 to my cart?",
    assistant_tools
)


MULTI-STEP TASK

User: Can you find wireless headphones, check if they're in stock, and add 2 to my cart?
--------------------------------------------------
 Tool: search_database
 Input: {"query": "wireless headphones", "category": "electronics"}
 Result: {"results": [{"id": "P001", "name": "Wireless Headphones", "price": 79.99}, {"id": "P002", "name": "...
 Tool: check_inventory
 Input: {"product_id": "P001"}
 Result: {"in_stock": true, "quantity_available": 50}...
 Tool: add_to_cart
 Input: {"product_id": "P001", "quantity": 2}
 Result: {"success": true, "message": "Added 2 item(s) to cart"}...
--------------------------------------------------
 Assistant: Perfect! I've successfully:

1. **Found wireless headphones** - "Wireless Headphones" priced at $79.99
2. **Checked inventory** - They are in stock with 50 units available
3. **Added to cart** - Successfully added 2 units to your cart

Your wireless headphones are ready in your cart! The total for 2 units would be $159.98 (2 × $7

### Real-World Tool Example: Data Transformation


In [34]:
data_extraction_tool = {
    "name": "extract_invoice_data",
    "description": "Extract structured data from invoice text.",
    "input_schema": {
        "type": "object",
        "properties": {
            "invoice_number": {"type": "string"},
            "date": {"type": "string", "description": "Invoice date in YYYY-MM-DD format"},
            "vendor": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "address": {"type": "string"}
                },
                "required": ["name"]
            },
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "quantity": {"type": "integer"},
                        "unit_price": {"type": "number"},
                        "total": {"type": "number"}
                    },
                    "required": ["description", "quantity", "unit_price", "total"]
                }
            },
            "subtotal": {"type": "number"},
            "tax": {"type": "number"},
            "total": {"type": "number"}
        },
        "required": ["invoice_number", "date", "vendor", "line_items", "total"]
    }
}

# Sample invoice text
invoice_text = """
INVOICE #2024-1234

From: Tech Supplies Inc.
123 Business Park, Suite 456
San Francisco, CA 94102

Date: December 3, 2024

ITEMS:
- Laptop Stand (x2) @ $45.00 each = $90.00
- USB-C Cable 6ft (x5) @ $12.99 each = $64.95  
- Wireless Mouse (x1) @ $29.99 each = $29.99

Subtotal: $184.94
Tax (8.5%): $15.72
TOTAL DUE: $200.66
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[data_extraction_tool],
    tool_choice={"type": "tool", "name": "extract_invoice_data"},
    messages=[
        {"role": "user", "content": f"Extract the data from this invoice:\n\n{invoice_text}"}
    ]
)

tool_use = next(block for block in response.content if block.type == "tool_use")
invoice_data = tool_use.input

print("📄 Extracted Invoice Data:")
print(json.dumps(invoice_data, indent=2))


📄 Extracted Invoice Data:
{
  "invoice_number": "2024-1234",
  "date": "2024-12-03",
  "vendor": {
    "name": "Tech Supplies Inc.",
    "address": "123 Business Park, Suite 456, San Francisco, CA 94102"
  },
  "line_items": [
    {
      "description": "Laptop Stand",
      "quantity": 2,
      "unit_price": 45.0,
      "total": 90.0
    },
    {
      "description": "USB-C Cable 6ft",
      "quantity": 5,
      "unit_price": 12.99,
      "total": 64.95
    },
    {
      "description": "Wireless Mouse",
      "quantity": 1,
      "unit_price": 29.99,
      "total": 29.99
    }
  ],
  "subtotal": 184.94,
  "tax": 15.72,
  "total": 200.66
}


<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>